In [1]:
import time
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean, sum as spark_sum, count

# Create Spark session configured for your 10-core M4
spark = SparkSession.builder \
    .appName("M4_Performance_Test") \
    .master("local[10]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Using {spark.sparkContext.defaultParallelism} cores\n")

# ===== EXAMPLE 1: Simple aggregation comparison =====
print("=" * 60)
print("EXAMPLE 1: Aggregating 10 million rows")
print("=" * 60)

# Create test data
n_rows = 10_000_000
data = {
    'id': range(n_rows),
    'value': np.random.randn(n_rows),
    'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], n_rows)
}

# Pandas approach (single-core)
df_pandas = pd.DataFrame(data)

start = time.time()
result_pandas = df_pandas.groupby('category')['value'].agg(['mean', 'sum', 'count'])
pandas_time = time.time() - start

print(f"\nPandas (single-core): {pandas_time:.2f} seconds")
print(result_pandas)

# PySpark approach (multi-core)
df_spark = spark.createDataFrame(df_pandas)

start = time.time()
result_spark = df_spark.groupBy('category').agg(
    mean('value').alias('mean'),
    spark_sum('value').alias('sum'),
    count('id').alias('count')
).toPandas()
spark_time = time.time() - start

print(f"\nPySpark (10-core): {spark_time:.2f} seconds")
print(result_spark.sort_values('category'))

speedup = pandas_time / spark_time
print(f"\nSpeedup: {speedup:.2f}x faster with PySpark")

# ===== EXAMPLE 2: Complex transformation =====
print("\n" + "=" * 60)
print("EXAMPLE 2: Complex calculations on 5 million rows")
print("=" * 60)

n_rows = 5_000_000
data2 = {
    'x': np.random.randn(n_rows),
    'y': np.random.randn(n_rows),
    'z': np.random.randn(n_rows)
}

df_pandas2 = pd.DataFrame(data2)

# Pandas approach
start = time.time()
df_pandas2['result'] = (df_pandas2['x'] ** 2 + 
                        df_pandas2['y'] ** 2 + 
                        df_pandas2['z'] ** 2) ** 0.5
df_pandas2['normalized'] = df_pandas2['result'] / df_pandas2['result'].max()
pandas_stats = df_pandas2['normalized'].describe()
pandas_time2 = time.time() - start

print(f"\nPandas: {pandas_time2:.2f} seconds")

# PySpark approach
df_spark2 = spark.createDataFrame(df_pandas2[['x', 'y', 'z']])

start = time.time()
from pyspark.sql.functions import sqrt, pow as spark_pow, max as spark_max

df_spark2 = df_spark2.withColumn(
    'result', 
    sqrt(spark_pow(col('x'), 2) + spark_pow(col('y'), 2) + spark_pow(col('z'), 2))
)

max_val = df_spark2.agg(spark_max('result')).collect()[0][0]
df_spark2 = df_spark2.withColumn('normalized', col('result') / max_val)
spark_stats = df_spark2.select('normalized').describe().toPandas()
spark_time2 = time.time() - start

print(f"PySpark: {spark_time2:.2f} seconds")

speedup2 = pandas_time2 / spark_time2
print(f"\nSpeedup: {speedup2:.2f}x faster with PySpark")

# Clean up
spark.stop()
print("\n" + "=" * 60)
print("Spark session stopped")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/25 11:30:39 WARN Utils: Your hostname, alpamayo.local, resolves to a loopback address: 127.0.0.1; using 10.0.1.21 instead (on interface en0)
26/01/25 11:30:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/25 11:30:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Using 10 cores

EXAMPLE 1: Aggregating 10 million rows

Pandas (single-core): 0.34 seconds
              mean          sum    count
category                                
A        -0.000971 -1943.100490  2000937
B        -0.000347  -693.979956  1998852
C        -0.000587 -1175.747131  2001905
D        -0.000255  -509.157488  1998655
E        -0.000254  -507.275773  1999651


26/01/25 11:31:41 WARN TaskSetManager: Stage 0 contains a task of very large size (17485 KiB). The maximum recommended task size is 1000 KiB.
                                                                                


PySpark (10-core): 2.55 seconds
  category      mean          sum    count
4        A -0.000971 -1943.100490  2000937
1        B -0.000347  -693.979956  1998852
3        C -0.000587 -1175.747131  2001905
2        D -0.000255  -509.157488  1998655
0        E -0.000254  -507.275773  1999651

Speedup: 0.13x faster with PySpark

EXAMPLE 2: Complex calculations on 5 million rows

Pandas: 0.17 seconds


26/01/25 11:32:11 WARN TaskSetManager: Stage 3 contains a task of very large size (14174 KiB). The maximum recommended task size is 1000 KiB.
26/01/25 11:32:11 WARN TaskSetManager: Stage 6 contains a task of very large size (14174 KiB). The maximum recommended task size is 1000 KiB.


PySpark: 1.19 seconds

Speedup: 0.14x faster with PySpark

Spark session stopped
